In [4]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [5]:
# Strategy parameters

RTH_START = "08:30"
RTH_END = "14:55"

In [6]:
DATA_PATH = Path("../data/nq-1m_bk.csv")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    header=None,
    names=[
        "Date",
        "Time",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ],
)

df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M",
)

df = (
    df.drop(columns=["Date", "Time"])
      .sort_values("Datetime")
      .reset_index(drop=True)
)

df = df[
    [
        "Datetime",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ]
]

print(f"Rows loaded: {len(df):,}")

Rows loaded: 5,884,752


In [7]:
# Keep only the research period

START_DATE = "2022-07-01"

df = df[df["Datetime"] >= START_DATE].copy()

print(f"Rows: {len(df):,}")
print(
    f"Date range: {df['Datetime'].min()} → {df['Datetime'].max()}"
)

Rows: 1,414,163
Date range: 2022-07-01 00:00:00 → 2026-07-01 00:00:00


In [8]:
# Assign futures trading sessions (17:00 CT rollover)

df["SessionDate"] = (
    df["Datetime"]
    + pd.to_timedelta((df["Datetime"].dt.hour >= 17).astype(int), unit="D")
).dt.date

print(f"Unique sessions: {df['SessionDate'].nunique():,}")
print(
    f"Date range: {df['SessionDate'].min()} to {df['SessionDate'].max()}"
)

Unique sessions: 1,034
Date range: 2022-07-01 to 2026-07-01


In [9]:
def compute_daily_atr(df):
    daily = (
        df.groupby("SessionDate")
        .agg(
            Open=("Open", "first"),
            High=("High", "max"),
            Low=("Low", "min"),
            Close=("Close", "last"),
        )
    )

    daily["PrevClose"] = daily["Close"].shift(1)

    daily["TR"] = pd.concat(
        [
            daily["High"] - daily["Low"],
            (daily["High"] - daily["PrevClose"]).abs(),
            (daily["Low"] - daily["PrevClose"]).abs(),
        ],
        axis=1,
    ).max(axis=1)

    daily["ATR14"] = daily["TR"].rolling(14).mean().shift(1)

    return daily


daily = compute_daily_atr(df)

df["ATR14"] = df["SessionDate"].map(daily["ATR14"])

# Remove sessions without an ATR
df = df[df["ATR14"].notna()].copy()

print(f"Unique sessions: {df['SessionDate'].nunique():,}")
print(f"Rows: {len(df):,}")


Unique sessions: 1,020
Rows: 1,395,503


In [10]:
# Keep only Regular Trading Hours (Chicago time)

rth = df.set_index("Datetime").between_time(
    RTH_START,
    RTH_END,
).reset_index()

print(f"RTH rows: {len(rth):,}")
print(f"Sessions: {rth['SessionDate'].nunique():,}")


RTH rows: 385,964
Sessions: 1,016


In [15]:
IMPULSE_ATR = 0.1

TARGET_MULTIPLE = 0.7
STOP_MULTIPLE = 1.0

In [16]:
def first_impulse(rth, impulse_size):
    """
    Returns:
        direction      : "Up", "Down", "Unknown", or "Neither"
        impulse_index  : index of the candle that first completed the impulse
    """

    opening_price = rth.iloc[0]["Open"]

    up_target = opening_price + impulse_size
    down_target = opening_price - impulse_size

    for idx, candle in rth.iterrows():
        hit_up = candle["High"] >= up_target
        hit_down = candle["Low"] <= down_target

        if hit_up and hit_down:
            return "Unknown", idx

        if hit_up:
            return "Up", idx

        if hit_down:
            return "Down", idx

    return "Neither", None

In [17]:
trades = []
unknowns = []

for session_date, session in rth.groupby("SessionDate"):
    atr = session.iloc[0]["ATR14"]
    impulse_size = atr * IMPULSE_ATR

    direction, impulse_index = first_impulse(session, impulse_size)

    if direction in ["Unknown", "Neither"]:
        unknowns.append({
            "SessionDate": session_date,
            "Direction": direction,
            "ATR14": atr,
            "ImpulseSize": impulse_size,
        })
        continue

    impulse_candle = session.loc[impulse_index]

    trades.append({
        "SessionDate": session_date,
        "Direction": direction,
        "ATR14": atr,
        "ImpulseSize": impulse_size,
        "EntryTime": impulse_candle["Datetime"],
        "EntryPrice": (
            session.iloc[0]["Open"] + impulse_size
            if direction == "Up"
            else session.iloc[0]["Open"] - impulse_size
        ),
        "ImpulseIndex": impulse_index,
    })

trades = pd.DataFrame(trades)
unknowns = pd.DataFrame(unknowns)

print(f"Trades   : {len(trades)}")
print(f"Unknowns : {len(unknowns)}")

Trades   : 1005
Unknowns : 11


In [18]:
results = []

for _, trade in trades.iterrows():
    session = rth[rth["SessionDate"] == trade["SessionDate"]]

    # Only candles after the impulse candle
    future = session.loc[trade["ImpulseIndex"] + 1 :]

    entry = trade["EntryPrice"]
    risk = trade["ImpulseSize"]

    if trade["Direction"] == "Up":
        stop = entry - STOP_MULTIPLE * risk
        target = entry + TARGET_MULTIPLE * risk

        outcome = "Open"

        for _, candle in future.iterrows():
            hit_stop = candle["Low"] <= stop
            hit_target = candle["High"] >= target

            if hit_stop and hit_target:
                outcome = "Ambiguous"
                break
            elif hit_stop:
                outcome = "Loss"
                break
            elif hit_target:
                outcome = "Win"
                break

    else:
        stop = entry + STOP_MULTIPLE * risk
        target = entry - TARGET_MULTIPLE * risk

        outcome = "Open"

        for _, candle in future.iterrows():
            hit_stop = candle["High"] >= stop
            hit_target = candle["Low"] <= target

            if hit_stop and hit_target:
                outcome = "Ambiguous"
                break
            elif hit_stop:
                outcome = "Loss"
                break
            elif hit_target:
                outcome = "Win"
                break

    results.append({
        **trade.to_dict(),
        "Outcome": outcome,
        "Stop": stop,
        "Target": target,
    })

results = pd.DataFrame(results)

summary = results["Outcome"].value_counts()

summary["Unknown"] = (unknowns["Direction"] == "Unknown").sum()
summary["Neither"] = (unknowns["Direction"] == "Neither").sum()

summary = summary.reindex(
    ["Win", "Loss", "Ambiguous", "Open", "Unknown", "Neither"],
    fill_value=0,
)

display(summary.to_frame("Count"))

print()
print(f"Total sessions: {summary.sum()}")

,Count
Outcome,
Win,583
Loss,412
Ambiguous,3
Open,7
Unknown,2
Neither,9



Total sessions: 1016
